In [7]:
!pip install -q --upgrade --force-reinstall \
  "numpy" \
  "pandas" \
  "datasets" \
  "transformers" \
  "evaluate" \
  "accelerate" \
  "scikit-learn" \
  "sentencepiece"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 165.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 175.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.

In [8]:
!pip install -q --force-reinstall --no-deps \
  "torch" \
  "torchvision" \
  "torchaudio"

In [1]:
import numpy as np
import pandas as pd
import torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

numpy: 2.4.3
pandas: 3.0.1
cuda available: True
gpu: NVIDIA H100 80GB HBM3


In [2]:
import random

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import evaluate

In [3]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

import ctypes
ctypes.CDLL("/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0")
print("loaded successfully")

loaded successfully


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [6]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

dialog: a list of string features.
act: a list of classification labels, with possible values including __dummy__ (0), inform (1), question (2), directive (3) and commissive (4).
emotion: a list of classification labels, with possible values including no emotion (0), anger (1), disgust (2), fear (3), happiness (4), sadness (5) and surprise (6).

In [7]:
ds = load_dataset("pixelsandpointers/better_daily_dialog")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
print(ds)
print(ds["train"][0])
print(ds["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 87170
    })
    validation: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 8069
    })
    test: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 7740
    })
})
{'dialog_id': 0, 'utterance': 'Say , Jim , how about going for a few beers after dinner ? ', 'turn_type': 3, 'emotion': 0}
['dialog_id', 'utterance', 'turn_type', 'emotion']


In [9]:
def flatten_dailydialog(split):
    rows = {
        "utterance": [],
        "context_2": [],
        "context_3": [],
        "label": [],
        "dialog_id": [],
        "turn_id": [],
    }

    # group by dialog_id since data is already flat
    from itertools import groupby
    keyfunc = lambda x: x["dialog_id"]
    sorted_split = sorted(split, key=keyfunc)

    for dialog_id, turns in groupby(sorted_split, key=keyfunc):
        turns = list(turns)
        utterances = [t["utterance"] for t in turns]
        acts = [t["turn_type"] for t in turns]  # turn_type = acts in this dataset

        for i, utt in enumerate(utterances):
            current_speaker = "[S1]" if i % 2 == 0 else "[S2]"

            prev2 = []
            for j in range(max(0, i - 2), i):
                spk = "[S1]" if j % 2 == 0 else "[S2]"
                prev2.append(f"{spk} {utterances[j]}")
            context_2 = " ".join(prev2 + [f"{current_speaker} {utt}"])

            prev3 = []
            for j in range(max(0, i - 3), i):
                spk = "[S1]" if j % 2 == 0 else "[S2]"
                prev3.append(f"{spk} {utterances[j]}")
            context_3 = " ".join(prev3 + [f"{current_speaker} {utt}"])

            rows["utterance"].append(utt)
            rows["context_2"].append(context_2)
            rows["context_3"].append(context_3)
            rows["label"].append(acts[i] - 1)  # convert 1-4 -> 0-3
            rows["dialog_id"].append(dialog_id)
            rows["turn_id"].append(i)

    return Dataset.from_dict(rows)

flat_ds = DatasetDict({
    "train": flatten_dailydialog(ds["train"]),
    "validation": flatten_dailydialog(ds["validation"]),
    "test": flatten_dailydialog(ds["test"]),
})

In [10]:
id2label = {
    0: "inform",
    1: "question",
    2: "directive",
    3: "commissive"
}
label2id = {v: k for k, v in id2label.items()}
print(id2label)

{0: 'inform', 1: 'question', 2: 'directive', 3: 'commissive'}


In [11]:
for split_name in ["train", "validation", "test"]:
    counts = pd.Series(flat_ds[split_name]["label"]).value_counts().sort_index()
    print(f"\n{split_name.upper()}")
    for i, count in counts.items():
        print(f"{id2label[i]}: {count}")


TRAIN
inform: 39873
question: 24974
directive: 14242
commissive: 8081

VALIDATION
inform: 3125
question: 2244
directive: 1775
commissive: 925

TEST
inform: 3534
question: 2210
directive: 1278
commissive: 718


In [12]:
X_train = flat_ds["train"]["utterance"]
y_train = flat_ds["train"]["label"]

X_val = flat_ds["validation"]["utterance"]
y_val = flat_ds["validation"]["label"]

X_test = flat_ds["test"]["utterance"]
y_test = flat_ds["test"]["label"]

print("Train examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

Train examples: 87170
Validation examples: 8069
Test examples: 7740


In [20]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    lowercase=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (87170, 20000)
Validation TF-IDF shape: (8069, 20000)
Test TF-IDF shape: (7740, 20000)


In [22]:
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logreg.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [23]:
val_preds = logreg.predict(X_val_tfidf)
test_preds = logreg.predict(X_test_tfidf)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation Macro F1:", f1_score(y_val, val_preds, average="macro"))

print("\nTest Accuracy:", accuracy_score(y_test, test_preds))
print("Test Macro F1:", f1_score(y_test, test_preds, average="macro"))

Validation Accuracy: 0.728714834551989
Validation Macro F1: 0.7017653894843251

Test Accuracy: 0.739405684754522
Test Macro F1: 0.6904924141194213


In [24]:
print(classification_report(
    y_test,
    test_preds,
    target_names=[id2label[i] for i in range(4)]
))

              precision    recall  f1-score   support

      inform       0.83      0.72      0.77      3534
    question       0.89      0.83      0.86      2210
   directive       0.61      0.72      0.66      1278
  commissive       0.38      0.61      0.47       718

    accuracy                           0.74      7740
   macro avg       0.68      0.72      0.69      7740
weighted avg       0.77      0.74      0.75      7740



In [25]:
cm = confusion_matrix(y_test, test_preds)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{id2label[i]}" for i in range(4)],
    columns=[f"pred_{id2label[i]}" for i in range(4)]
)
cm_df

,pred_inform,pred_question,pred_directive,pred_commissive
true_inform,2528,115,350,541
true_question,178,1841,144,47
true_directive,160,86,916,116
true_commissive,171,19,90,438


In [26]:
error_df = pd.DataFrame({
    "utterance": X_test,
    "true_label": [id2label[y] for y in y_test],
    "pred_label": [id2label[p] for p in test_preds]
})

errors = error_df[error_df["true_label"] != error_df["pred_label"]].copy()
errors.head(25)

,utterance,true_label,pred_label
2,"Weed ! You know ? Pot , Ganja , Mary Jane som...",directive,inform
4,I also have blow if you prefer to do a few li...,directive,inform
10,Yeah ?,question,inform
20,"Well , we use the exhaust gases from our prin...",inform,directive
28,"Yes , I believe there are green teas , black ...",question,inform
32,"Sure , I like drinking tea at teahouses .",inform,commissive
33,"Oh , so do I .",inform,question
44,"Yes , I like travelling . I am young , and un...",inform,commissive
48,you ’ re right . We will be able to travel at...,inform,commissive
50,"ok . You haven ’ t seen my company car , have...",directive,question


In [51]:
checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [52]:
def tokenize_utterance(batch):
    return tokenizer(
        batch["utterance"],
        truncation=True,
        max_length=96
    )

tok_utt = flat_ds.map(tokenize_utterance, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]

tok_utt = tok_utt.remove_columns(
    [c for c in tok_utt["train"].column_names if c not in cols_to_keep]
)

tok_utt = tok_utt.rename_column("label", "labels")
tok_utt.set_format("torch")

Map:   0%|          | 0/87170 [00:00<?, ? examples/s]

Map:   0%|          | 0/8069 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

In [53]:
model_utt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./roberta_utt_debug",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=False,
    bf16=False,
    report_to="none"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [56]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_utt = Trainer(
    model=model_utt,
    args=training_args,
    train_dataset=tok_utt["train"],
    eval_dataset=tok_utt["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_utt.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.513943,0.529387,0.795018,0.744645
2,0.426141,0.509269,0.809642,0.757129
3,0.360287,0.527905,0.813979,0.762913


TrainOutput(global_step=16347, training_loss=0.4334570808338304, metrics={'train_runtime': 502.3659, 'train_samples_per_second': 520.557, 'train_steps_per_second': 32.54, 'total_flos': 5811090839926944.0, 'train_loss': 0.4334570808338304, 'epoch': 3.0})

In [57]:
from collections import Counter

pred_output = trainer_utt.predict(tok_utt["validation"])
preds = pred_output.predictions.argmax(axis=-1)

print("Predicted label counts:", Counter(preds.tolist()))
print("True label counts:", Counter(list(tok_utt["validation"]["labels"])))

Predicted label counts: Counter({0: 3314, 1: 2299, 2: 1758, 3: 698})
True label counts: Counter({tensor(1): 1, tensor(0): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(1): 1, tensor(2): 1, tensor(1): 1, tensor(2): 1, tensor(3): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(2): 1, tensor(1): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(2): 1, tensor(3): 1, tensor(2): 1, tensor(3): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(3): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(

In [58]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = list(tok_utt["validation"]["labels"])
y_pred = preds.tolist()

print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(4)],
    digits=4
))

print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

      inform     0.8048    0.8534    0.8284      3125
    question     0.9269    0.9496    0.9381      2244
   directive     0.7651    0.7577    0.7614      1775
  commissive     0.6089    0.4595    0.5237       925

    accuracy                         0.8140      8069
   macro avg     0.7764    0.7551    0.7629      8069
weighted avg     0.8076    0.8140    0.8092      8069

[[2667   49  218  191]
 [  12 2131   96    5]
 [ 254   99 1345   77]
 [ 381   20   99  425]]


BERTweet

In [14]:
checkpoint_bt = "vinai/bertweet-base"

# BERTweet often works more smoothly with use_fast=False
tokenizer_bt = AutoTokenizer.from_pretrained(checkpoint_bt, use_fast=False)

def tokenize_utterance_bt(batch):
    return tokenizer_bt(
        batch["utterance"],
        truncation=True,
        max_length=96
    )

tok_utt_bt = flat_ds.map(tokenize_utterance_bt, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]

tok_utt_bt = tok_utt_bt.remove_columns(
    [c for c in tok_utt_bt["train"].column_names if c not in cols_to_keep]
)

tok_utt_bt = tok_utt_bt.rename_column("label", "labels")
tok_utt_bt.set_format("torch")

model_utt_bt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_bt,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics_bt(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

data_collator_bt = DataCollatorWithPadding(tokenizer=tokenizer_bt)

training_args_bt = TrainingArguments(
    output_dir="./bertweet_utt_debug",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=False,
    bf16=False,
    report_to="none"
)


Map:   0%|          | 0/87170 [00:00<?, ? examples/s]

Map:   0%|          | 0/8069 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

In [15]:
trainer_utt_bt = Trainer(
    model=model_utt_bt,
    args=training_args_bt,
    train_dataset=tok_utt_bt["train"],
    eval_dataset=tok_utt_bt["validation"],
    data_collator=data_collator_bt,
    compute_metrics=compute_metrics_bt
)

trainer_utt_bt.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.499225,0.509089,0.804809,0.755876
2,0.406655,0.502580,0.811253,0.759793
3,0.335888,0.529275,0.816086,0.766113


TrainOutput(global_step=16347, training_loss=0.4139226347676562, metrics={'train_runtime': 806.6561, 'train_samples_per_second': 324.19, 'train_steps_per_second': 20.265, 'total_flos': 5564835025677408.0, 'train_loss': 0.4139226347676562, 'epoch': 3.0})

In [16]:
pred_output_bt = trainer_utt_bt.predict(tok_utt_bt["validation"])
preds_bt = pred_output_bt.predictions.argmax(axis=-1)

from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix

y_true_bt = list(tok_utt_bt["validation"]["labels"])
y_pred_bt = preds_bt.tolist()

print("Predicted label counts:", Counter(y_pred_bt))
print("True label counts:", Counter(int(x) for x in y_true_bt))

print(classification_report(
    y_true_bt,
    y_pred_bt,
    target_names=[id2label[i] for i in range(4)],
    digits=4
))

print(confusion_matrix(y_true_bt, y_pred_bt))

Predicted label counts: Counter({0: 3309, 1: 2314, 2: 1740, 3: 706})
True label counts: Counter({0: 3125, 1: 2244, 2: 1775, 3: 925})
              precision    recall  f1-score   support

      inform     0.8078    0.8554    0.8309      3125
    question     0.9235    0.9523    0.9377      2244
   directive     0.7701    0.7549    0.7624      1775
  commissive     0.6161    0.4703    0.5334       925

    accuracy                         0.8161      8069
   macro avg     0.7794    0.7582    0.7661      8069
weighted avg     0.8097    0.8161    0.8114      8069

[[2673   47  210  195]
 [  15 2137   87    5]
 [ 253  111 1340   71]
 [ 368   19  103  435]]
